# Ligand Graph Builder



## Relevant packages

### Pytorch Geometric (PyG)
We use PyG, a library built upon PyTorch to easily write and train Graph Neural Networks (GNNs) to generate our ligand graphs. 

In [ ]:
# Install all libraries
! pip install pytorch-lightning wandb rdkit ogb
# install PyG
! pip install torch_geometric

`Might not need this!`Set a random seed to ensure repeatability of experiments

In [ ]:
import random
import numpy as np
import torch

# Random seeds and reproducibility
torch.manual_seed(0)
torch.cuda.manual_seed(0)
np.random.seed(0)
random.seed(0)

## Graph representation

For our project, we use a molecular graph to represent the chromophores. The **nodes** are the heavy atoms of the chromophore (i.e. hydrogen atoms are not considered as nodes). We extract the chromophore atoms from the resolved three dimensional structure of the fluorescent protein in the Protein Data Bank. 
We then build an RDKit molecule from the coordinates so that we can build a molecular graph using atom and bond features from RDKit. 
...As this with **node positions** given by the raw atom euclidian xyz extracted from the resolved crystal structure. The **edges** are the euclidian distances between atoms.

## (Alternative Graph representation)

For our project, we use a molecular graph to represent the chromophores. The **nodes** are the heavy atoms of the chromophore (i.e. hydrogen atoms are not considered as nodes) and the **edges** are the chemical bonds. 

In order to get both the correct bond orders but also retain the 3D geometry of the chromophore in the protein environment (**node/edge features**), we use a hybrid approach: 
1. We use the PDB chromophore abbreviation to obtain the SMILES of the chromophores as a chemical skeleton 
2. We align the SMILES to the atomic coordinates in the resolved three dimensional structure in the PDB using RDKit's `AssignBondOrdersFromTemplate`
3. We build the graph from the hybrid molecule

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import Draw
import requests

IPythonConsole.ipython_useSVG = True  # < use SVGs instead of PNGs
IPythonConsole.drawOptions.addAtomIndices = True  # adding indices for atoms
IPythonConsole.drawOptions.addBondIndices = False  # not adding indices for bonds
IPythonConsole.molSize = 200, 200

In [ ]:
import re
from pathlib import Path

CHROMOPHORE_CODES = {
    'NRQ', 'CRQ', 'NRP', 'CH6', 'CRO', '5SQ', '4M9', 'CR2', 'OFM', 'CR8',
    'CFY', 'OIM', 'CH7', 'GYS', 'WCR', 'GYC', 'DYG', 'FAD', 'PIA', 'CCY',
    'BLR', 'CRF', 'NYG', 'CR7', 'FMN', 'B2H', 'SWG', 'CSH', 'BJF'
}

def get_chromophore_code(pdb_path: str | Path) -> str | None: 
    """
    Extract the chromophore residue code from a PDB legacy file. Scans HETATM records and returns the first residue name that matches
    a known chromophore code.
    Args:
        pdb_path: Path to the .pdb file.
    Returns:
        The matched residue code (e.g. 'CRO'), or None if not found.
    """
    pdb_path = Path(pdb_path)
    if not pdb_path.exists():
        raise FileNotFoundError(f"PDB file not found: {pdb_path}")

    with open(pdb_path, 'r') as f:
        for line in f:
            if not line.startswith('HETATM'):
                continue
            # PDB format: columns 18-20 (0-indexed 17:20) = residue name
            residue_name = line[17:20].strip()
            if residue_name in CHROMOPHORE_CODES:
                return residue_name

    return None  # no chromophore found

def get_chromophore_pdb_block(
    pdb_path: str | Path,
    residue_code: str,
    chain_id: str | None = None
) -> str:
    """
    Extract all HETATM lines for a given chromophore residue code, formatted as a PDB block ready for RDKit's MolFromPDBBlock.
    Args:
        pdb_path:     Path to the .pdb file.
        residue_code: The chromophore residue name (e.g. 'CRO').
        chain_id:     Optional chain ID to filter by (e.g. 'A'). If None, defaults to the first chain containing the chromophore.
    Returns:
        A string PDB block containing only the chromophore's HETATM lines.
    Raises:
        ValueError: If the residue code is not found, or if the specified chain does not contain the chromophore.
    """
    pdb_path = Path(pdb_path)
    
    # PDB column positions (0-indexed):
    # [17:20] residue name, [21] chain ID, [22:26] residue sequence number

    # --- Pass 1: collect all (chain, resseq) pairs for this chromophore ---
    occurrences: dict[str, set[str]] = {}  # chain_id → set of residue seq numbers
    with open(pdb_path, 'r') as f:
        for line in f:
            if not line.startswith('HETATM'):
                continue
            if line[17:20].strip() != residue_code:
                continue
            chain  = line[21].strip()
            resseq = line[22:26].strip()
            occurrences.setdefault(chain, set()).add(resseq)

    if not occurrences:
        raise ValueError(
            f"Residue '{residue_code}' not found in {pdb_path.name}."
        )

    # --- Resolve chain ---
    if chain_id is None:
        # Default to the first chain encountered (alphabetical for determinism)
        chain_id = sorted(occurrences.keys())[0]
        if len(occurrences) > 1:
            print(
                f"[INFO] '{residue_code}' found in chains "
                f"{sorted(occurrences.keys())} in {pdb_path.name}. "
                f"Defaulting to chain '{chain_id}'. "
                f"Pass chain_id=... to override."
            )
    elif chain_id not in occurrences:
        raise ValueError(
            f"Residue '{residue_code}' not found in chain '{chain_id}' "
            f"of {pdb_path.name}. "
            f"Available chains: {sorted(occurrences.keys())}."
        )

    # --- Pass 2: extract HETATM lines for the resolved chain ---
    lines = []
    with open(pdb_path, 'r') as f:
        for line in f:
            if not line.startswith('HETATM'):
                continue
            if line[17:20].strip() != residue_code:
                continue
            if line[21].strip() != chain_id:
                continue
            lines.append(line)

    return ''.join(lines) + 'END\n'

In [ ]:
def get_ccd_smiles(residue_code: str) -> str: 
    """Fetch canonical SMILES from the RCSB Chemical Component Dictionary."""
    url = f"https://data.rcsb.org/rest/v1/core/chemcomp/{residue_code}"
    data = requests.get(url).json()
    return data["rcsb_chem_comp_descriptor"]["smiles"]

def chromophore_mol_from_pdb(pdb_block: str, residue_code: str) -> Chem.Mol:
    """
    Build a chemically correct, 3D-positioned chromophore molecule.
    Combines CCD bond orders with real PDB coordinates.
    """
    # 1. Get the chemically correct template from CCD
    smiles = get_ccd_smiles(residue_code)
    template = Chem.MolFromSmiles(smiles)

    # 2. Parse the raw PDB block (coordinates only, bonds unreliable)
    raw_mol = Chem.MolFromPDBBlock(pdb_block, sanitize=False, removeHs=False)

    # 3. Assign correct bond orders from the template onto the 3D structure
    mol = AllChem.AssignBondOrdersFromTemplate(template, raw_mol)
    mol = Chem.RemoveHs(mol)  # optional: heavy-atom-only graph
    Chem.SanitizeMol(mol)
    return mol

Add a line of code here that executes all these steps? Or somewhere else?

## Graph construction from the hybrid molecule

We first define the **node and edge features** (atom and bond features)

We use the following **node features** (4-dim) extracted from RDKit:  

| feature | description |
| ---- | ----  |
| atom type  | atomic number |
| degree  | number of directly-bonded neighbor atoms, including H atoms |
| formal charge | integer electronic charge assigned to atom |
| hybridization | sp, sp2, sp3, sp3d, or sp3d2 |

**Edge features** (3-dim) are the following:  

| feature | description |
| ---- | ----  |
| bond type  | single, double, triple, or aromatic |
| stereo | none, any, E/Z or cis/trans |
| conjugated  | whether the bond is conjugated |

In [ ]:
ATOM_FEATURES = {
    'atom_type' : [1, 6, 7, 8, 9, 16],  # elements: H, C, N, O, F
    'degree' : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'formal_charge' : [-5, -4, -3, -2, -1, 0, 1, 2, 3, 4, 5],
    'hybridization' : [
        'SP', 'SP2', 'SP3', 'SP3D', 'SP3D2'
        ],
}

BOND_FEATURES = {
    'bond_type':  ['SINGLE', 'DOUBLE', 'TRIPLE', 'AROMATIC'],
    'stereo':     ['STEREONONE', 'STEREOZ', 'STEREOE', 'STEREOCIS', 'STEREOTRANS', 'STEREOANY'],
    'conjugated': [False, True],
}

def get_atom_fv(atom, pos):
    """
    Converts rdkit atom object to a feature vector of indices + 3D position.
    Args: 
        atom: rdkit atom object
        pos:  np.array of shape (3,) with XYZ coordinates from PDB conformer
   Returns: 
        list of feature values
    """
    atom_fv = [
        ATOM_FEATURES['atom_type'].index(atom.GetAtomicNum()),
        ATOM_FEATURES['degree'].index(atom.GetTotalDegree()),
        ATOM_FEATURES['formal_charge'].index(atom.GetFormalCharge()),
        ATOM_FEATURES['hybridization'].index(str(atom.GetHybridization())),
        float(pos[0]),  # X coordinate (Å)
        float(pos[1]),  # Y coordinate (Å)
        float(pos[2]),  # Z coordinate (Å)
    ]
    return atom_fv

def get_bond_fv(bond):
    """
    Converts rdkit bond object to a feature vector
    Args: 
        bond:  rdkit bond object 
        pos_i: np.array of shape (3,) - position of begin atom
        pos_j: np.array of shape (3,) - position of end atom
    Returns: 
        list of feature values
    """
    dist = float(np.linalg.norm(pos_i - pos_j))
    bond_fv = [
        dist, 
        BOND_FEATURES['bond_type'].index(str(bond.GetBondType())),
        BOND_FEATURES['stereo'].index(str(bond.GetStereo())),
        BOND_FEATURES['conjugated'].index(bond.GetIsConjugated()),
    ]
    return bond_fv

In [ ]:
#DO WE NEED THIS!????
# Show indices of bonds
IPythonConsole.drawOptions.addAtomIndices = False  # not adding indices for atoms
IPythonConsole.drawOptions.addBondIndices = True  # adding indices for bonds
mol

### Molecular graph data

We use [Data](https://pytorch-geometric.readthedocs.io/en/latest/generated/torch_geometric.data.Data.html#torch_geometric.data.Data) class in `PyG` to create a graph data for DMF.

In [ ]:
from torch_geometric.data import Data

# convert our data to tensors, which are used for model training
x = torch.tensor(atom_fvs, dtype=torch.float)
edge_index = torch.tensor(edge_index, dtype=torch.long)
edge_attr = torch.tensor(bond_fvs, dtype=torch.float)

chromophore_data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)
chromophore_data

def mol_to_graph(mol: Chem.Mol, residue_name: str, protein_code: str) -> Data:
    """
    Converts a chromophore RDKit molecule to a PyTorch Geometric Data object.

    Args:
        mol:          RDKit molecule with 3D conformer (from PDB)
        residue_name: chromophore residue code (e.g. 'CRO')
        protein_code: PDB ID of the protein (e.g. '1EMA')

    Returns:
        torch_geometric.data.Data graph object
    """
    conf      = mol.GetConformer()
    positions = np.array([conf.GetAtomPosition(i) for i in range(mol.GetNumAtoms())])

    # --- Atom features ---
    atom_fvs = [get_atom_fv(atom, positions[i]) for i, atom in enumerate(mol.GetAtoms())]

    # --- Bond features + edge index ---
    edge_index0, edge_index1 = [], []
    bond_fvs = []
    for bond in mol.GetBonds():
        i, j  = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        edge_index0 += [i, j]
        edge_index1 += [j, i]
        fv = get_bond_fv(bond, positions[i], positions[j])
        bond_fvs += [fv, fv]  # one entry per direction

    edge_index = [edge_index0, edge_index1]

    # --- Label ---
    y = residue_name + "_" + protein_code  # e.g. 'CRO_1EMA'

    # --- Convert to tensors ---
    x          = torch.tensor(atom_fvs,   dtype=torch.float)
    edge_index = torch.tensor(edge_index, dtype=torch.long)
    edge_attr  = torch.tensor(bond_fvs,   dtype=torch.float)

    chromophore_data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)
    return chromophore_data

## 2.2 Graph Neural Network

A [graph neural network (GNN)](https://en.wikipedia.org/wiki/Graph_neural_network) is a class of artificial neural networks for processing data that can be represented as graphs. GNNs rely on [message passing methods](https://arxiv.org/abs/1704.01212), which means that nodes exchange information with the neighbors, and send "messages" to each other. Generally, GNNs operate in two phases: a **message passing** phase, which transmits information across the molecule to build a neural representation of the molecule, and a **readout** phase, which uses the final representation of the molecule to make predictions about the properties of interest.

### Message passing

Before looking at the math, we can try to visually understand how message passing works. The first step is that each node creates a `feature vector` that represents the `message` it wants to send to all its neighbors. In the second step, the messages are sent to the neighbors, so that a node receives one message per adjacent node. As shown in the figure below, after a message passing step, `node 1` can get the message from `node 2`, and `node 2` can get messages from `node 1`, `node 3` and `node 4`. The third step is that each node will aggregate all messages from neighbors and get a `message vector`. Then, the fourth step is that each node updates its `feature vector` based on its `message vector` and previous `feature vector`.

<center width="100%" style="padding:10px"><img src="img/graph_message_passing.svg" width="700px"></center>

Moreover, with the iteration of message passing, each node can obtain the feature vectors of more distant nodes and not limited to neighbors. As shown in the figure below, node `A` can get informations from node `E` and node `F` in the interation 2, which are not the neighbors of node `A`.  Node `C`, the neighbor of node `A`, can obtain the information of nodes `E` and `F` in the iteration 1, so node `A` can obtain the information of nodes `E` and `F` in the iteration 2.

<center width="100%" style="padding:10px"><img src="img/messages.svg" width="700px"></center>

**You can also read the mathematical formulas to better understand the process of message passing.**

1) Initialization

Get initial hidden feature vector $h_i^0$ of node $i$ from its original node features $x_i$
$$
h_i^0 = I(x_i), \quad \forall i \in V
\tag{1}
$$
where $I$ is initialize function

2) Send message
$$
m_{j \rightarrow i}^{t+1} = M(h_i^t, \: h_j^t, \: e_{i,j})
\tag{2}
$$
where $m_{j \rightarrow i}^{t+1}$ is the message sent from node $j$ to $i$ at the $t+1$ iteration, $M$ is message function, and $e_{i,j}$ is the feature of edge between node $i$ and $j$

3) Message aggregation
$$
m_i^{t+1} = \sum_{j \in N(i)} m_{j \rightarrow i}^{t+1}
\tag{3}
$$
where $N(i)$ presents all neighbor nodes of node $i$, and $m_i^{t+1}$ is the aggregated message of node $i$ at the $t+1$ iteration

4) Node update
$$
h_i^{t+1} = U(h_i^t, \: m_i^{t+1})
\tag{4}
$$
where $h_i^t$ is the hidden feature vector of node $i$ at the $t$ iteration, and $U$ is the update function

### Readout
The readout layers will aggregate the hidden feature vectors of all nodes and get graph-level vectors (that is, the properties we want to predict).

$$
\hat{y} = R(\{ h_i^T \: | \: i \in V\})
\tag{5}
$$
where $h_i^T$ is the final hidden feature vector of node $i$, $\: \: \hat{y}$ is graph-level vectors (our prediction target), and $R$ is the readout function

**Note that in GNNs, the $I$, $M$, $U$ and $R$ functions need to be differentiable, such as multi-layer artificial neural networks.**

### Code example

Here, we will define a GNN model using message passing neural network (MPNN) according to paper ["Neural Message Passing for Quantum Chemistry"](https://arxiv.org/abs/1704.01212). We just use [NNConv](https://pytorch-geometric.readthedocs.io/en/latest/generated/torch_geometric.nn.conv.NNConv.html#torch_geometric.nn.conv.NNConv) class to create message passing layers of our models. The [torch_geometric.nn](https://pytorch-geometric.readthedocs.io/en/latest/modules/nn.html) module of PyG contains many different types of layers for message passing and readout, which can help us define GNN models more conveniently.

In [ ]:
import torch
import torch.nn.functional as F
from torch.nn import GRU
import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger
from torch_geometric.loader import DataLoader
from torch_geometric.nn import NNConv, MLP, global_add_pool
from ogb.graphproppred.mol_encoder import AtomEncoder, BondEncoder


class MPNN(pl.LightningModule):
    """
    Message Passing Neural Network

    Node update procedure:
        1. Encode atoms -> node embeddings
        2. Encode bonds -> edge embeddings
        3. Edge features generate weight matrices (NNConv)
        4. Messages are aggregated from neighbors
        5. GRU updates node states
    """
    def __init__(
        self,
        node_embedding_dim,     # dimension of node embeddings (H)
        output_dim,             # graph prediction dimension
        train_data,
        valid_data,
        test_data,
        std,
        batch_size=32,
        lr=1e-3,
        num_message_steps=3
    ):
        super().__init__()
        self.std = std  # std of data's target
        self.train_data = train_data
        self.valid_data = valid_data
        self.test_data = test_data
        self.batch_size = batch_size
        self.lr = lr
        self.num_message_steps = num_message_steps
        # Input encoders
        self.atom_emb = AtomEncoder(emb_dim=node_embedding_dim)
        self.bond_emb = BondEncoder(emb_dim=node_embedding_dim)
        # Edge conditioned message function
        edge_network_output_dim = node_embedding_dim * node_embedding_dim

        self.edge_network = MLP([
            node_embedding_dim,            # edge embedding size
            2 * node_embedding_dim,        # hidden layer
            edge_network_output_dim        # produces H² values
        ])
        self.message_layer = NNConv(
            in_channels=node_embedding_dim,
            out_channels=node_embedding_dim,
            nn=self.edge_network,
            aggr="mean"
        )
        # Node update function
        self.node_update = GRU(
            input_size=node_embedding_dim,
            hidden_size=node_embedding_dim
        )
        # Graph level readout
        self.graph_mlp = MLP([
            node_embedding_dim,
            node_embedding_dim // 2,
            output_dim
        ])

    def forward(self, data):
        # Initialization
        node_state = self.atom_emb(data.x)
        edge_attr = self.bond_emb(data.edge_attr)
        hidden_state = node_state.unsqueeze(0)
        
        # Message passing
        for step in range(self.num_message_steps):
            messages = self.message_layer(
                node_state,
                data.edge_index,
                edge_attr
            )

            messages = F.relu(messages)

            # Node state update
            node_state, hidden_state = self.node_update(
                messages.unsqueeze(0),
                hidden_state
            )
            node_state = node_state.squeeze(0)

        # Graph readout - combine node embeddings into graph embedding
        graph_embedding = global_add_pool(node_state, data.batch)
        prediction = self.graph_mlp(graph_embedding)

        return prediction.view(-1)
        
    def training_step(self, batch, batch_idx):
        pred = self(batch)
        loss = F.mse_loss(pred, batch.y)
        self.log("train_loss", loss)
        return loss

    def validation_step(self, batch, batch_idx):
        pred = self(batch)
        mse = F.mse_loss(pred * self.std, batch.y * self.std)
        self.log("valid_mse", mse)

    def test_step(self, batch, batch_idx):
        pred = self(batch)
        mse = F.mse_loss(pred * self.std, batch.y * self.std)
        self.log("test_mse", mse)

    def configure_optimizers(self):
        # Here we configure the optimization algorithm.
        optimizer = torch.optim.Adam(
            self.parameters(),
            lr=self.lr
        )
        return optimizer
    
    def train_dataloader(self):
        return DataLoader(self.train_data, batch_size=self.batch_size, shuffle=True)
    
    def val_dataloader(self):
        return DataLoader(self.valid_data, batch_size=self.batch_size, shuffle=False)
    
    def test_dataloader(self):
        return DataLoader(self.test_data, batch_size=self.batch_size, shuffle=False)

Here, we can use [InMemoryDataset]() class in PyG to create the graph dataset of ESOL conveniently. You can also browse its [tutorial](https://pytorch-geometric.readthedocs.io/en/latest/tutorial/create_dataset.html) and [pre-defined dataset](https://pytorch-geometric.readthedocs.io/en/latest/modules/datasets.html) to learn about how to create graph datasets quickly by PyG.

In [ ]:
from tqdm import tqdm
import pandas as pd
import torch
from torch_geometric.data import (
    Data,
    InMemoryDataset,
    download_url,
)
from ogb.utils import smiles2graph


class ESOLGraphData(InMemoryDataset):
    """The ESOL graph dataset using PyG
    """
    # ESOL dataset download link
    raw_url = 'https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/delaney-processed.csv'

    def __init__(self, root, transform=None):
        super().__init__(root, transform)
        self.data, self.slices = torch.load(self.processed_paths[0], weights_only=False)

    @property
    def raw_file_names(self):
        return ['delaney-processed.csv']

    @property
    def processed_file_names(self):
        return ['data.pt']

    def download(self):
        print('Downloading ESOL dataset...')
        file_path = download_url(self.raw_url, self.raw_dir)

    def process(self):
        # load raw data from a csv file
        df = pd.read_csv(self.raw_paths[0])
        smiles = df['smiles'].values.tolist()
        target = df['measured log solubility in mols per litre'].values.tolist()

        # Convert SMILES into graph data
        print('Converting SMILES strings into graphs...')
        data_list = []
        for i, smi in enumerate(tqdm(smiles)):

            # get graph data from SMILES
            graph = smiles2graph(smi)

            # convert to tensor and pyg data
            x = torch.tensor(graph['node_feat'], dtype=torch.long)
            edge_index = torch.tensor(graph['edge_index'], dtype=torch.long)
            edge_attr = torch.tensor(graph['edge_feat'], dtype=torch.long)
            y = torch.tensor([target[i]], dtype=torch.float)
            data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)
            data_list.append(data)

        # save data
        torch.save(self.collate(data_list), self.processed_paths[0])

Create, normalize and split ESOL graph dataset.

In [ ]:
from typing import Optional, Tuple
import numpy as np
from torch_geometric.data.dataset import Dataset


class RandomSplitter(object):
    """Class for doing random data splits."""

    def split(
        self,
        dataset: Dataset,
        frac_train: float = 0.7,
        frac_valid: float = 0.1,
        frac_test: float = 0.2,
        seed: Optional[int] = None,
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        """
        Splits internal compounds randomly into train/validation/test.

        Parameters
        ----------
        dataset: Dataset
          Dataset to be split.
        seed: int, optional (default None)
          Random seed to use.
        frac_train: float, optional (default 0.8)
          The fraction of data to be used for the training split.
        frac_valid: float, optional (default 0.1)
          The fraction of data to be used for the validation split.
        frac_test: float, optional (default 0.1)
          The fraction of data to be used for the test split.
        seed: int, optional (default None)
          Random seed to use.

        Returns
        -------
        Tuple[np.ndarray, np.ndarray, np.ndarray]
          A tuple of train indices, valid indices, and test indices.
          Each indices is a numpy array.
        """
        np.testing.assert_almost_equal(frac_train + frac_valid + frac_test, 1.0)
        if seed is not None:
            np.random.seed(seed)
        num_datapoints = len(dataset)
        train_cutoff = int(frac_train * num_datapoints)
        valid_cutoff = int((frac_train + frac_valid) * num_datapoints)
        shuffled = np.random.permutation(range(num_datapoints))
        return (
            shuffled[:train_cutoff],
            shuffled[train_cutoff:valid_cutoff],
            shuffled[valid_cutoff:],
        )

In [ ]:
# create dataset
dataset = ESOLGraphData('./esol_pyg').shuffle()

# Normalize target to mean = 0 and std = 1.
mean = dataset.data.y.mean()
std = dataset.data.y.std()
dataset.data.y = (dataset.data.y - mean) / std
mean, std = mean.item(), std.item()

# split data
splitter = RandomSplitter()
train_idx, valid_idx, test_idx = splitter.split(dataset, frac_train=0.7, frac_valid=0.1, frac_test=0.2)
train_dataset = dataset[train_idx]
valid_dataset = dataset[valid_idx]
test_dataset = dataset[test_idx]

In [ ]:
# This will ask you to login to your wandb account
import wandb

wandb.init(project="gnn-solubility",
           config={
               "batch_size": 32,
               "learning_rate": 0.001,
               "hidden_size": 64,
               "max_epochs": 60
           })

Train and evaluate the model.

In [ ]:
# Here we create an instance of our GNN.
# Play around with the hyperparameters!
gnn_model = MPNN(
    node_embedding_dim=wandb.config["hidden_size"],
    output_dim=1,
    train_data=train_dataset,
    valid_data=valid_dataset,
    test_data=test_dataset,
    std=std,
    batch_size=wandb.config["batch_size"],
    lr=wandb.config["learning_rate"]
)

# Define trainer: How we want to train the model
wandb_logger = WandbLogger()
trainer = pl.Trainer(
    max_epochs = wandb.config["max_epochs"],
    logger = wandb_logger
)

# Finally! Training a model :)
trainer.fit(
    model=gnn_model,
)

# Now run test
results = trainer.test(ckpt_path="best")
wandb.finish()

# Test RMSE
test_mse = results[0]["test_mse"]
test_rmse = test_mse ** 0.5
print(f"\nMPNN model performance: RMSE on test set = {test_rmse:.4f}.\n")